In [0]:
# Notebook 04: 04_model_registration_mlflow.ipynb (Versão Final Limpa)

import mlflow
import os 
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col, count
from pyspark.ml.functions import vector_to_array 
from mlflow.models.signature import infer_signature 

# --- 0. CONFIGURAÇÃO DO VOLUME UC PARA ARTEFATOS TEMPORÁRIOS ---
UC_VOLUME_PATH_BASE = "/Volumes/bigdata_anomaly_detection_kddcup99_catalogue/default/kdd_volume"
MLFLOW_TEMP_PATH = f"{UC_VOLUME_PATH_BASE}/mlflow_temp"
os.environ['MLFLOW_DFS_TMP'] = MLFLOW_TEMP_PATH
print(f"MLFLOW_DFS_TMP definido como: {MLFLOW_TEMP_PATH}")

# --- 1. Configurações de Dados ---
BASE_TABLE_NAME = "bigdata_anomaly_detection_kddcup99_catalogue.default"
TRAIN_TABLE_NAME = f"{BASE_TABLE_NAME}.kdd_features_train"
TEST_TABLE_NAME = f"{BASE_TABLE_NAME}.kdd_features_test"

df_train = spark.table(TRAIN_TABLE_NAME)
df_test = spark.table(TEST_TABLE_NAME)

# --- 2. Configurações MLflow (Unity Catalog) ---
UC_MODEL_NAME = f"{BASE_TABLE_NAME}.kdd_intrusion_detection_rf"
EXPERIMENT_NAME = "/Shared/kdd_anomaly_detection_experiment"
mlflow.set_experiment(EXPERIMENT_NAME)

rf = RandomForestClassifier(
    labelCol="is_anomaly", 
    featuresCol="features", 
    numTrees=20, 
    maxDepth=5, 
    seed=42
)

# Inicia a execução do MLflow
with mlflow.start_run(run_name="RF_Final_PCA_Scaled") as run:
    print("\nIniciando o Treinamento e Registro MLflow...")
    
    rf_model = rf.fit(df_train)
    
    # 3. Log de Parâmetros e Modelo (MLflow)
    mlflow.log_param("num_trees", rf.getNumTrees())
    mlflow.log_param("max_depth", rf.getMaxDepth())
    mlflow.log_param("n_features_pca", 10)
    
    # 4. Avaliação e Log de Métricas
    df_predictions = rf_model.transform(df_test)
    evaluator_auc = BinaryClassificationEvaluator(labelCol="is_anomaly", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    mlflow.log_metric("test_auc", evaluator_auc.evaluate(df_predictions))

    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="is_anomaly", predictionCol="prediction", metricName="f1")
    mlflow.log_metric("test_f1_score", evaluator_f1.evaluate(df_predictions))
    mlflow.log_metric("test_accuracy", evaluator_f1.evaluate(df_predictions, {evaluator_f1.metricName: "accuracy"}))

    # 5. Registro do Modelo no Unity Catalog
    print("\nRegistrando o modelo no Unity Catalog via MLflow...")
    
    # Prepara o input_example e infere a assinatura
    df_input_example = df_test.limit(1).withColumn(
        "features", 
        vector_to_array(col("features"))
    ).select("features")
    
    model_signature = infer_signature(
        df_input_example.toPandas(), 
        df_predictions.select("prediction").limit(1).toPandas()
    )
    
    # Registra o modelo
    run_info = mlflow.spark.log_model(
        spark_model=rf_model, 
        artifact_path="random_forest_model",
        registered_model_name=UC_MODEL_NAME,
        signature=model_signature, 
        input_example=df_input_example.toPandas(),
        await_registration_for=30
    )
    
    # Simplificamos a saída final para usar o run_info (que retorna a versão do modelo logado)
    print(f"\n✅ PROJETO CONCLUÍDO: Modelo registrado com sucesso no UC.")
    print(f"Nome do Modelo: {UC_MODEL_NAME}")
    print(f"Versão: {run_info.model_version}")